# Fine-tune Flan-T5 with LoRA on SAMSum (dialogue summarization)


In [ ]:
# Install the libraries we need. transformers for the model, peft for LoRA,
# datasets for loading from the Hub, accelerate for training loop utils.
!pip install -q transformers peft datasets accelerate huggingface_hub

In [ ]:
!pip install -q -U torchao

In [ ]:
!pip install -q rouge_score

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
import torch

In [ ]:
# Load SAMSum. Columns are already named "dialogue" and "summary" —
# no custom extraction function needed, unlike the job-description dataset.
dataset = load_dataset("knkarthick/samsum")

print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})
{'id': '13818513', 'dialogue': "Amanda: I baked  cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)", 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}


In [ ]:
# For a 1-2 hour experience, work with a small subset instead of the full
# ~14,700 training examples. This keeps training time short while still
# giving you a real, visibly-improving model.
train_data = dataset["train"].shuffle(seed=42).select(range(800))
val_data = dataset["validation"].shuffle(seed=42).select(range(100))

print(f"Train size: {len(train_data)}, Val size: {len(val_data)}")

Train size: 800, Val size: 100


In [ ]:
# Load base model and tokenizer. flan-t5-base is a good size for a free T4 GPU.
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
# Configure LoRA. rank=8 keeps the adapter small and fast to train.
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,                        # rank of the LoRA update matrices
    lora_alpha=32,              # scaling factor
    lora_dropout=0.1,
    target_modules=["q", "v"],  # attention query and value projections
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # sanity check: should be a small % of total params

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


In [ ]:
# Tokenize the dataset. Prefix the input so the model learns the task framing.
PREFIX = "Summarize this conversation: "
MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 128

def tokenize_batch(batch):
    inputs = [PREFIX + text for text in batch["dialogue"]]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LEN, truncation=True)

    labels = tokenizer(
        text_target=batch["summary"], max_length=MAX_TARGET_LEN, truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tokenized = train_data.map(tokenize_batch, batched=True, remove_columns=train_data.column_names)
val_tokenized = val_data.map(tokenize_batch, batched=True, remove_columns=val_data.column_names)

In [ ]:
# Data collator handles padding per batch instead of padding the whole dataset upfront
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [ ]:
# Training arguments. 1 epoch on 800 examples
# fine-tuning
training_args = Seq2SeqTrainingArguments(
    output_dir="./lora-samsum",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-4,
    num_train_epochs=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    predict_with_generate=True,
    fp16=False,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [ ]:
# Train. On a T4 with flan-t5-base and 800 examples for 1 epoch,
# expect roughly 10-15 minutes.
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.496569,1.470546


TrainOutput(global_step=100, training_loss=1.465030975341797, metrics={'train_runtime': 78.3855, 'train_samples_per_second': 10.206, 'train_steps_per_second': 1.276, 'total_flos': 377885815971840.0, 'train_loss': 1.465030975341797, 'epoch': 1.0})

In [ ]:
# Quick manual test — compare the base model's instinct against your fine-tuned one.
test_dialogue = """Amanda: I baked cookies. Do you want some?
Jerry: Sure! Bring me some tomorrow.
Amanda: I will leave you some in the fridge.
Jerry: You're the best!"""

test_input = PREFIX + test_dialogue
inputs = tokenizer(test_input, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_length=100)
print("Fine-tuned summary:", tokenizer.decode(output[0], skip_special_tokens=True))

# Compare against the actual reference summary from the dataset, e.g. for a similar example:
print("\nA few real reference summaries for comparison:")
for i in range(3):
    print("-", dataset["train"][i]["summary"])

Fine-tuned summary: Amanda baked cookies. Jerry will bring them to Amanda tomorrow.

A few real reference summaries for comparison:
- Amanda baked cookies and will bring Jerry some tomorrow.
- Olivia and Olivier are voting for liberals in this election. 
- Kim may try the pomodoro technique recommended by Tim to get more stuff done.


In [ ]:
# %%
# Load a separate, untouched copy of the base model for comparison.
# (Your `model` variable already has the LoRA adapter merged in from training,
# so we load a fresh instance here rather than reusing it.)
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

inputs = tokenizer(test_input, return_tensors="pt").to(base_model.device)
base_output = base_model.generate(**inputs, max_length=100)

print("Base model (no fine-tuning):", tokenizer.decode(base_output[0], skip_special_tokens=True))
print("Fine-tuned model:", tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Base model (no fine-tuning): Amanda baked cookies. Jerry will bring them to Amanda tomorrow.
Fine-tuned model: Amanda baked cookies. Jerry will bring them to Amanda tomorrow.


In [ ]:
from rouge_score import rouge_scorer
import numpy as np

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def generate_summary(model_to_use, dialogue_text, max_len=100):
    text = PREFIX + dialogue_text
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(model_to_use.device)
    out = model_to_use.generate(**inputs, max_length=max_len)
    return tokenizer.decode(out[0], skip_special_tokens=True)

# Load a fresh, untouched base model — `model` already has the LoRA adapter merged in.
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

base_scores = {"rouge1": [], "rouge2": [], "rougeL": []}
finetuned_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

# Run over the full validation set (100 examples) — takes a couple of minutes on a T4.
for example in val_data:
    dialogue = example["dialogue"]
    reference = example["summary"]

    base_summary = generate_summary(base_model, dialogue)
    finetuned_summary = generate_summary(model, dialogue)

    base_result = scorer.score(reference, base_summary)
    finetuned_result = scorer.score(reference, finetuned_summary)

    for key in base_scores:
        base_scores[key].append(base_result[key].fmeasure)
        finetuned_scores[key].append(finetuned_result[key].fmeasure)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
print("Average ROUGE F1 scores across 100 validation examples:\n")
print(f"{'Metric':<10} {'Base model':<15} {'Fine-tuned':<15} {'Difference'}")
for key in base_scores:
    base_avg = np.mean(base_scores[key])
    ft_avg = np.mean(finetuned_scores[key])
    diff = ft_avg - base_avg
    print(f"{key:<10} {base_avg:<15.4f} {ft_avg:<15.4f} {diff:+.4f}")

Average ROUGE F1 scores across 100 validation examples:

Metric     Base model      Fine-tuned      Difference
rouge1     0.4406          0.4749          +0.0343
rouge2     0.2147          0.2237          +0.0090
rougeL     0.3705          0.3887          +0.0182


In [ ]:
# Check whether the LoRA adapter actually has nonzero trained weights
for name, param in model.named_parameters():
    if "lora_B" in name:
        print(name, "‖weights‖ =", param.norm().item())
        break

base_model.model.encoder.block.0.layer.0.SelfAttention.q.lora_B.default.weight ‖weights‖ = 0.21603645384311676


In [ ]:
# Log in to Hugging Face so push_to_hub works. This will prompt for a token —
# generate one at huggingface.co/settings/tokens with "write" access.
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
for log in trainer.state.log_history:
    if "loss" in log:
        print(log)

{'loss': 1.451650905609131, 'grad_norm': 0.5666572451591492, 'learning_rate': 0.000243, 'epoch': 0.2, 'step': 20}
{'loss': 1.4560166358947755, 'grad_norm': 0.7071353197097778, 'learning_rate': 0.00018299999999999998, 'epoch': 0.4, 'step': 40}
{'loss': 1.5025032043457032, 'grad_norm': 0.5091643333435059, 'learning_rate': 0.00012299999999999998, 'epoch': 0.6, 'step': 60}
{'loss': 1.4184154510498046, 'grad_norm': 0.6437386274337769, 'learning_rate': 6.299999999999999e-05, 'epoch': 0.8, 'step': 80}
{'loss': 1.4965686798095703, 'grad_norm': 0.5413147211074829, 'learning_rate': 2.9999999999999997e-06, 'epoch': 1.0, 'step': 100}


In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('finetune'))

In [ ]:
REPO_NAME = "Sasmita03/lora-samsum-summarizer"

model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)

print(f"Adapter pushed to https://huggingface.co/{REPO_NAME}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 9.86kB / 3.56MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Adapter pushed to https://huggingface.co/Sasmita03/lora-samsum-summarizer
